# Reversión – Eliminación de la arquitectura Medallion
Elimina las **tablas lógicas** de Unity Catalog y sus **rutas físicas** en el Data Lake, capa por capa.

| Capa | Tablas |
|---|---|
| bronze | `car_sales`, `car_features`, `dealers` |
| silver | `car_features_clean`, `car_sales_enriched` |
| golden | `ventas_mensuales_estado`, `ranking_modelos`, `kpi_concesionarios`, `segmentos_clientes` |

> Las tablas son **externas**: `DROP TABLE` solo elimina la metadata, por eso además se borran los archivos con `dbutils.fs.rm`.
> El container `raw` (archivos fuente) no se modifica.

In [0]:
%python
dbutils.widgets.removeAll()

In [0]:
%python
dbutils.widgets.text("storageName", "adlsjgl99")
dbutils.widgets.text("catalogo", "vehiculos")

In [0]:
%python
storageName = dbutils.widgets.get("storageName")
catalogo = dbutils.widgets.get("catalogo")

def ruta(capa, tabla):
    return f"abfss://{capa}@{storageName}.dfs.core.windows.net/{tabla}"

### Eliminación tablas Bronze

In [0]:
-- DROP TABLES
DROP TABLE IF EXISTS ${catalogo}.bronze.car_sales;
DROP TABLE IF EXISTS ${catalogo}.bronze.car_features;
DROP TABLE IF EXISTS ${catalogo}.bronze.dealers;

In [0]:
%python
## REMOVE DATA (Bronze)
dbutils.fs.rm(ruta("bronze", "car_sales"), True)
dbutils.fs.rm(ruta("bronze", "car_features"), True)
dbutils.fs.rm(ruta("bronze", "dealers"), True)

### Eliminación tablas Silver

In [0]:
-- DROP TABLES
DROP TABLE IF EXISTS ${catalogo}.silver.car_features_clean;
DROP TABLE IF EXISTS ${catalogo}.silver.car_sales_enriched;

In [0]:
%python
## REMOVE DATA (Silver)
dbutils.fs.rm(ruta("silver", "car_features_clean"), True)
dbutils.fs.rm(ruta("silver", "car_sales_enriched"), True)

### Eliminación tablas Golden

In [0]:
-- DROP TABLES
DROP TABLE IF EXISTS ${catalogo}.golden.ventas_mensuales_estado;
DROP TABLE IF EXISTS ${catalogo}.golden.ranking_modelos;
DROP TABLE IF EXISTS ${catalogo}.golden.kpi_concesionarios;
DROP TABLE IF EXISTS ${catalogo}.golden.segmentos_clientes;

In [0]:
%python
## REMOVE DATA (Golden)
dbutils.fs.rm(ruta("golden", "ventas_mensuales_estado"), True)
dbutils.fs.rm(ruta("golden", "ranking_modelos"), True)
dbutils.fs.rm(ruta("golden", "kpi_concesionarios"), True)
dbutils.fs.rm(ruta("golden", "segmentos_clientes"), True)

### Validación

In [0]:
SHOW TABLES IN ${catalogo}.bronze;

In [0]:
SHOW TABLES IN ${catalogo}.silver;

In [0]:
SHOW TABLES IN ${catalogo}.golden;

In [0]:
%python
for capa in ["bronze", "silver", "golden"]:
    contenido = dbutils.fs.ls(f"abfss://{capa}@{storageName}.dfs.core.windows.net/")
    print(f"{capa}: {len(contenido)} carpetas restantes")

### Reversión completa (opcional)
Elimina además los esquemas, el catálogo y las external locations. Se deja comentado para no borrar la infraestructura por accidente;
*1.Preparacion_Ambiente* la vuelve a crear.

In [0]:
-- DROP SCHEMA IF EXISTS ${catalogo}.raw;
-- DROP SCHEMA IF EXISTS ${catalogo}.bronze;
-- DROP SCHEMA IF EXISTS ${catalogo}.silver;
-- DROP SCHEMA IF EXISTS ${catalogo}.golden;
-- DROP CATALOG IF EXISTS ${catalogo} CASCADE;

In [0]:
-- DROP EXTERNAL LOCATION IF EXISTS `exlt-raw`;
-- DROP EXTERNAL LOCATION IF EXISTS `exlt-bronze`;
-- DROP EXTERNAL LOCATION IF EXISTS `exlt-silver`;
-- DROP EXTERNAL LOCATION IF EXISTS `exlt-golden`;
-- DROP EXTERNAL LOCATION IF EXISTS `exlt-metastore`;